# Kysynnän ajureiden löytäminen PROC GLMSELECTilla: askeltava valinta, LASSO ja validoitu eteenpäin-valinta

## Tiivistelmä

Vähittäiskaupan analytiikkatiimi käyttää **PROC GLMSELECTia** selvittääkseen, mitkä markkinoinnin ja hinnoittelun vivut todella liikuttavat viikoittaista yksikkömyyntiä, erottaen todelliset kysynnän ajurit kohinasta. SBC-kriteerillä pisteytetty askeltava valinta, 5-kertaisella ristiinvalidoinnilla tehty LASSO ja validointiotoksella validoitu eteenpäin-haku päätyvät kaikki **samaan joukkoon todellisia ajureita** — oma hinta, kilpailijan hinta, mainoskulut, sähköpostien määrä, kampanjat, juhlapyhät, koillinen alue sekä verkkokanava — ja jokainen neljästä istutetusta harhautimesta (`temp_f`, `noise1`, `noise2`, `noise3`) hylätään automaattisesti.

Menetelmät ovat myös suuruusluokiltaan hyvin samaa mieltä: kukin arvioi oman hinnan vaikutukseksi noin **-28 yksikköä dollaria kohti** ja kilpailijan hinnan vaikutukseksi noin **+14** — juuri ne substituutioetumerkit, jotka datan generoiva yhtälö sisälsi. Ainoa erimielisyys on marginaalissa — ristiinvalidoitu LASSO pitää lisäksi mukana pienen, tilastollisesti merkityksettömän `Region=Midwest`-kontrastin (estimaatti -8,3, keskivirhe 8,3), jonka sekä askeltava valinta että eteenpäin-valinta hylkäävät. Ajurilista, joka kestää kolme erilaista valintafilosofiaa, on paljon luotettavampi kuin yhteen sääntöön viritetty.

## Tietolähteet

Kaikki tämän muistikirjan data on **synteettistä** ja generoitu suoraan koodissa (ei ulkoisia tiedostoja, siemenluku `20260531`). Se jäljittelee yhtä myyntikautta kulutustavaroiden vähittäiskauppiaan myymälä-viikko-paneelista.

| Aineisto | Rivit | Taso | Keskeiset muuttujat |
|---------|------|-------|---------------|
| `demand` | 100 | Myymälä-viikko | `units` (vaste: viikoittain myydyt yksiköt); `price`, `competitor` (oma ja kilpailijan hyllyhinta); `adspend`, `email` (mediapaine); `promo`, `holiday` (tapahtumaliput); `region`, `channel` (CLASS-efektit); `temp_f`, `noise1`-`noise3` (harhautin/epäolennaiset selittäjät) |

`units` on rakennettu tunnetusta lineaarisesta yhtälöstä, jotta "oikea" ajurijoukko on todennettavissa; `temp_f` ja kolme `noise`-saraketta eivät kanna todellista signaalia ja niiden tarkoitus on testata, hylkääkö jokainen valintamenetelmä ne.

# Kysynnän ajureiden löytäminen PROC GLMSELECTilla

Tuoteryhmäpäälliköllä on kymmeniä ehdokasselittäjiä viikoittaiselle myynnille: oma hinta, kilpailijan hinta, mainostuskulut, sähköpostien määrä, kampanjat, juhlapyhät, myymälän alue, myyntikanava, jopa sää. Kaikkien syöttäminen yhteen regressioon houkuttelee ylisovittamiseen ja epävakaisiin kertoimiin. **PROC GLMSELECT** automatisoi säästeliään mallin haun tukien askeltavaa, LASSO-, elastic net- ja least-angle-valintaa sisäänrakennetulla ristiinvalidoinnilla ja validointiositoksella.

Tässä muistikirjassa:

1. Generoimme realistisen synteettisen myymälä-viikko-kysyntäpaneelin, jossa on *tunnettu* joukko todellisia ajureita (sekä tarkoituksella lisättyjä harhauttimia).
2. Ajamme **askeltavan valinnan**, jota pisteytetään SBC:llä.
3. Ajamme **LASSOn** 5-kertaisella ristiinvalidoinnilla.
4. Ajamme **eteenpäin-valinnan**, joka validoidaan 30 %:n validointiotoksella.

Hyvän valintamenetelmän pitäisi löytää todelliset ajurit ja hylätä kohina — katsotaanpa.

## 1. Generoi synteettinen kysyntäpaneeli

Vaste `units` rakennetaan eksplisiittisestä lineaarisesta yhtälöstä, joten tiedämme perustotuuden: hinta ja kilpailijan hinta, mainoskulut, sähköpostien määrä, kampanja- ja juhlapyhäliput sekä alue- ja kanavaefektit vaikuttavat kaikki. Muuttujat `temp_f`, `noise1`, `noise2` ja `noise3` ovat puhtaita harhauttimia ilman todellista yhteyttä myyntiin. `call streaminit`-siemenluku tekee datasta toistettavan.

In [1]:
/* ---------------------------------------------------------------
   Synteettinen myymälä-viikko-kysyntäpaneeli kulutustavaroiden
   vähittäiskauppiaalle. 'units' noudattaa TUNNETTUA yhtälöä;
   temp_f ja noise1-3 ovat harhauttimia.
   --------------------------------------------------------------- */
TIEDOT demand;
    CALL streaminit(20260531);
    PITUUS region $9 channel $8 promo $3;
    TEE store_week = 1 ASTI 100;
        /* Aluejakauma */
        u1 = rand('uniform');
        JOS u1 < 0.34 NIIN region = 'Northeast';
        MUUTEN JOS u1 < 0.67 NIIN region = 'Midwest';
        MUUTEN region = 'South';

        /* Myyntikanava */
        JOS rand('uniform') < 0.45 NIIN channel = 'Store';
        MUUTEN channel = 'Online';

        /* Hinnoittelu- ja mediaeffektit */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Tapahtumaliput ja epäolennainen säähäirikkö */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        JOS rand('uniform') < 0.40 NIIN promo = 'Yes';
        MUUTEN promo = 'No';

        /* Puhtaat kohintaselittäjät (ei todellista vaikutusta) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Viikon myyntiyksiköt tunnetusta rakenneyhtälöstä */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        JOS units < 0 NIIN units = 0;
        TULOSTE;
    LOPPU;
    SÄILYTÄ region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
SUORITA;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Datan profilointi

Ennen mallinnusta varmistetaan, että vaste ja tärkeimmät jatkuvat ehdokasmuuttujat ovat järkevällä asteikolla.

In [2]:
PROSEDUURI KESKIARVOT TIEDOT=demand n mean std MIN MAX maxdec=1;
    MUUTTUJA units price competitor adspend email;
    NIMIKE units="Viikkomyynti (kpl)" price="Oma hinta" competitor="Kilpailijan hinta"
          adspend="Mainoskulut" email="Sähköpostien määrä";
    OTSIKKO "Viikoittainen kysyntä ja ehdokasajurit";
SUORITA;

                                         Viikoittainen kysyntä ja ehdokasajurit                                         

                                                  The MEANS Procedure

 Variable    Label                           N        Mean     Std Dev     Minimum     Maximum
 ---------------------------------------------------------------------------------------------
 units       Viikkomyynti (kpl)            100       874.8       136.3       508.6      1162.3
 price       Oma hinta                     100        14.0         3.4         8.0        19.9
 competitor  Kilpailijan hinta             100        13.8         3.4         8.1        19.9
 adspend     Mainoskulut                   100       990.6       726.9        50.0      3358.0
 email       Sähköpostien määrä            100        45.5        26.4         0.0        99.0
 ---------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Askeltava valinta pisteytettynä SBC:llä

Askeltava valinta lisää ja poistaa efektejä vuorotellen, tässä arvioituna **Schwarzin bayesilaisella kriteerillä (SBC)** sekä sisäänottotestissä (`select=sbc`) että lopullisen mallin valinnassa (`choose=sbc`). SBC rankaisee monimutkaisuudesta AIC:tä ankarammin ja suosii siten karumpia malleja.

Keskeiset lauseet ja optiot:

- `CLASS region channel promo / param=reference` määrittää kategoriset selittäjät referenssisolukoodauksella.
- `selection=stepwise(select=sbc choose=sbc)` ohjaa hakua.
- `details=summary` tulostaa askel askeleelta -valintayhteenvedon; `stb` lisää standardoidut kertoimet, jotta eri asteikoilla olevia efektejä voi verrata.
- `output out=demand_scored p=predicted r=residual` tallentaa sovitetut arvot ja jäännökset jatkopisteytystä varten.

Lue **Stepwise Selection Summary** hakujälkenä: se lähtee täydestä 12 efektin mallista ja *poistaa* efektejä yksi kerrallaan, pudottaen vuorollaan `noise1`:n, `noise2`:n, `temp_f`:n, `Region=Midwest`-kontrastin ja `noise3`:n, koska jokainen poisto laskee SBC:tä. **Parameter Estimates** -taulukkoon jäänyt on valittu malli.

In [3]:
PROSEDUURI glmselect TIEDOT=demand seed=20260531;
    LUOKKA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(VALITSE=sbc choose=sbc)
          details=summary stb;
    TULOSTE out=demand_scored p=predicted r=residual;
    NIMIKE region="Alue" channel="Kanava" promo="Kampanja" price="Oma hinta"
          competitor="Kilpailijan hinta" adspend="Mainoskulut"
          email="Sähköpostien määrä" temp_f="Lämpötila (F)"
          holiday="Juhlapyhä" noise1="Kohina1" noise2="Kohina2"
          noise3="Kohina3" units="Viikkomyynti (kpl)";
    OTSIKKO "Kysynnän ajureiden askeltava valinta (SBC)";
SUORITA;

                                         Viikoittainen kysyntä ja ehdokasajurit                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Viikkomyynti (kpl)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                 Stepwise Selection Summary                                                 

    Step    Action           Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ---------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed          Kohina1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
    


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO ristiinvalidoinnilla

LASSO kutistaa kertoimia kohti nollaa tehden valinnan ja regularisoinnin samanaikaisesti. Sen sijaan että pysähdyttäisiin kiinteään kriteeriin, annamme **5-kertaisen ristiinvalidoinnin** valita LASSO-polun kohdan, jolla on paras ennustevirhe otoksen ulkopuolella.

- `selection=lasso(choose=cv stop=none)` jäljittää koko LASSO-polun ja valitsee CV-optimaalisen askeleen.
- `cvmethod=random(5)` jakaa havainnot 5 satunnaiseen lohkoon.

**LASSO Selection Summary** näyttää järjestyksen, jossa efektit tulevat mukaan rangaistuksen löystyessä: `price` ensin, sitten `adspend`, `competitor`, koillinen alue, `email`, `promo` ja `holiday` — kaikki seitsemän todellista signaalia tulevat mukaan ennen yhtäkään harhautinta. Ristiinvalidointi valitsee sitten askeleen, jossa otoksen ulkopuolinen PRESS on pienin, ja tuloksena oleva **Parameter Estimates** -taulukko pitää täsmälleen aidot ajurit (plus pienen `Region=Midwest`-termin) sulkien pois `temp_f`:n, `noise1`:n, `noise2`:n ja `noise3`:n, jotka tulevat mukaan vasta aivan polun lopussa.

In [4]:
PROSEDUURI glmselect TIEDOT=demand seed=20260531;
    LUOKKA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv PYSÄYTÄ=none)
          cvmethod=RANDOM(5);
    NIMIKE region="Alue" channel="Kanava" promo="Kampanja" price="Oma hinta"
          competitor="Kilpailijan hinta" adspend="Mainoskulut"
          email="Sähköpostien määrä" temp_f="Lämpötila (F)"
          holiday="Juhlapyhä" noise1="Kohina1" noise2="Kohina2"
          noise3="Kohina3" units="Viikkomyynti (kpl)";
    OTSIKKO "LASSO 5-kertaisella ristiinvalidoinnilla";
SUORITA;

                                         Viikoittainen kysyntä ja ehdokasajurit                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Viikkomyynti (kpl)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                             LASSO Selection Summary                                                             

    Step    Action                   Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  --------------------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Eteenpäin-valinta validoituna validointiotoksella

Kolmas, täydentävä strategia: rakenna malli **eteenpäin-valinnalla** (efektit vain tulevat mukaan, eivät koskaan poistu), mutta pysäytä siihen, missä suoritus riippumattomalla validointiotoksella on paras — suojautuen suoraan ylisovittamiselta.

- `partition fraction(validate=0.30)` varaa satunnaisesti 30 % riveistä validointiin, jättäen 70 harjoitus- ja 30 validointihavaintoa.
- `selection=forward(select=aic choose=validate)` lisää efektejä AIC:n mukaan mutta valitsee lopullisen mallin validointiotoksen virheen perusteella.

**Partition Fractions** -taulukko vahvistaa 70/30-jaon. Eteenpäin-valinta lisää sitten efektejä, kunnes validointivirhe lakkaa parantumasta; lopullisen **Parameter Estimates** -taulukon kahdeksan efektiä ovat juuri ne todelliset ajurit, eivätkä neljä harhautinta koskaan pääse mukaan. Kun kolme eri periaatteille rakennettua menetelmää päätyy samoihin ajureihin, malli on paljon todennäköisemmin todellinen kuin minkään yksittäisen valintasäännön artefakti.

In [5]:
PROSEDUURI glmselect TIEDOT=demand seed=20260531;
    LUOKKA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(VALITSE=aic choose=validate);
    partition FRACTION(validate=0.30);
    NIMIKE region="Alue" channel="Kanava" promo="Kampanja" price="Oma hinta"
          competitor="Kilpailijan hinta" adspend="Mainoskulut"
          email="Sähköpostien määrä" temp_f="Lämpötila (F)"
          holiday="Juhlapyhä" noise1="Kohina1" noise2="Kohina2"
          noise3="Kohina3" units="Viikkomyynti (kpl)";
    OTSIKKO "Eteenpäin-valinta validoituna 30 %:n validointiotoksella";
SUORITA;

                                         Viikoittainen kysyntä ja ehdokasajurit                                         

The GLMSELECT Procedure


Dependent Variable: UNITS Viikkomyynti (kpl)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                            Forward Selection Summary                                                            

    Step    Action                   Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ---


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Tulosten tulkinta

Kaikki kolme valintastrategiaa löytävät **saman joukon todellisia kysynnän ajureita** ja hylkäävät jokaisen harhauttimen. Luettuna suoraan **Parameter Estimates** -taulukoista:

- **Hinta** on hallitseva ajuri. Sen standardoitu kerroin askeltavassa mallissa on **-0,70**, selvästi suurin suuruudeltaan, ja raaka kulmakerroin asettuu välille **-27,8** (askeltava ja LASSO) ja **-28,6** (eteenpäin) yksikköä dollaria kohti — lähes tarkalleen se -28, jolla data generoitiin. **Kilpailijan hinta** liikuttaa kysyntää toiseen suuntaan noin **+14,4**:llä kaikissa kolmessa sovituksessa — substituutiokäyttäytymistä, jota tuoteryhmäpäällikkö odottaa.
- **Mainoskulut** (noin +0,062 yksikköä dollaria kohti) ja **sähköpostien määrä** (noin +1,2 yksikköä lähetystä kohti) molemmat nostavat myyntiä ja kvantifioivat mediavasteen.
- **Kampanjoilla** ja **juhlapyhillä** on suurimmat tapahtumaefektit: `Promo=No`-kontrasti on noin **-51...-57** verrattuna kampanjaviikkoon, ja juhlapyhän nosto on noin **+66...+76** yksikköä. **Koillinen alue** (noin +49...+55) ja **verkkokanava** (noin +24...+32) kantavat positiivisia lähtötasoefektejä.
- Ratkaisevasti jokainen harhautin — `temp_f`, `noise1`, `noise2`, `noise3` — **pudotetaan** askeltavassa ja eteenpäin-valinnassa ja se suljetaan pois CV-valitusta LASSO-mallista. Jokainen menetelmä löysi rakenteellisen signaalin ja jätti kohinan huomiotta.

Kolme menetelmää eivät ole tavu tavulta identtisiä: askeltava (SBC) ja validointiotoksella validoitu eteenpäin-haku päätyvät samoihin kahdeksaan efektiin, kun taas ristiinvalidoitu LASSO pitää lisäksi mukana pienen `Region=Midwest`-kontrastin (-8,3, keskivirhe 8,3) — kohinatason ero eikä sisällöllinen erimielisyys. Se, että ajurilista kestää askeltavan SBC:n, ristiinvalidoidun LASSOn ja validointiotoksella validoidun eteenpäin-valinnan, on todellinen johtopäätös: kolmelle erilaiselle valintafilosofialle kestävä löydös on paljon uskottavampi kuin yhteen kriteeriin viritetty. `OUTPUT OUT=demand_scored` tallentaa sovitetut arvot ja jäännökset, ja sama työnkulku laajenee luontevasti ensi vuosineljänneksen suunniteltujen hinta- ja kampanjakalenterien pisteytykseen.